In [1]:
import torch

print(torch.__version__)

2.8.0+cu129


In [2]:
gpu_available = torch.cuda.is_available() # gpu 사용 가능 여부 확인
print(gpu_available)

True


In [3]:
gpu_count = torch.cuda.device_count() # 사용 가능한 GPU 개수 확인
gpu_count

1

In [4]:
current_device = torch.cuda.current_device() # 사용 가능한 장치 번호
current_device

0

In [5]:
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = True

In [6]:
from torchvision.datasets import FashionMNIST

fm_train = FashionMNIST(root='.', train=True, download=True)
fm_test = FashionMNIST(root='.', train=False, download=True)

In [7]:
type(fm_train.data)

torch.Tensor

In [8]:
print(fm_train.data.shape, fm_test.data.shape)

torch.Size([60000, 28, 28]) torch.Size([10000, 28, 28])


In [9]:
print(fm_train.targets.shape, fm_test.targets.shape)

torch.Size([60000]) torch.Size([10000])


In [10]:
fm_train

Dataset FashionMNIST
    Number of datapoints: 60000
    Root location: .
    Split: Train

In [11]:
fm_train.data[0]

tensor([[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   1,   0,
           0,  13,  73,   0,   0,   1,   4,   0,   0,   0,   0,   1,   1,   0],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   3,   0,
          36, 136, 127,  62,  54,   0,   0,   0,   1,   3,   4,   0,   0,   3],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   6,   0,
         102, 204, 176, 134, 144, 123,  23,   0,   0,   0,   0,  12,  10,   0],
        [  0,   0,   0,   0,   0,   0,   0,   

In [12]:
fm_train.targets[:10]

tensor([9, 0, 0, 3, 0, 2, 7, 2, 5, 5])

In [13]:
train_input = fm_train.data
train_target = fm_train.targets

In [14]:
train_scaled = train_input / 255.0

In [15]:
from sklearn.model_selection import train_test_split

train_scaled, val_scaled, train_target, val_target = train_test_split(train_scaled, train_target, test_size=0.2, random_state=42)

In [16]:
print(train_scaled.shape, val_scaled.shape)

torch.Size([48000, 28, 28]) torch.Size([12000, 28, 28])


### 케라스 모델과 파이토치 모델의 주요 차이점
1. 파이토치에서는 모델의 입력 크기를 사전에 지정할 필요가 없음
    - 따라서 케라스의 Input( )과 같은 별도의 입력 정의 함수가 없음
2. 케라스의 Dense 층과 동일한 역할을 하는 것이 파이토치의 Linear 층
    - Linear 층을 사용할 때는 입력 크기와 출력 크기(뉴런 개수)를 매개변수로 전달
3. 파이토치에서는 활성화 함수를 별도의 층으로 추가해야 합니다. 렐루 함수의 경우 ReLU 층을 사용
4. 출력층에 해당하는 두 번째 Linear 층 다음에는 활성화 함수가 없음
    - 케라스에서는 다중 분류 문제를 해결하기 위해 마지막 층에 소프트맥스 함수를 포함했지만, 파이토치에서는 이를 생략

In [17]:
import torch.nn as nn

model = nn.Sequential(
    nn.Flatten(), # 입력층
    nn.Linear(784,100), # 은닉층
    nn.ReLU(), # 활성함수
    nn.Linear(100,10) # 출력층
)

In [18]:
from torchinfo import summary

summary(model, input_size=(32,28,28)) # 32 : batch size, 28,28 : 이미지 사이즈

Layer (type:depth-idx)                   Output Shape              Param #
Sequential                               [32, 10]                  --
├─Flatten: 1-1                           [32, 784]                 --
├─Linear: 1-2                            [32, 100]                 78,500
├─ReLU: 1-3                              [32, 100]                 --
├─Linear: 1-4                            [32, 10]                  1,010
Total params: 79,510
Trainable params: 79,510
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 2.54
Input size (MB): 0.10
Forward/backward pass size (MB): 0.03
Params size (MB): 0.32
Estimated Total Size (MB): 0.45

In [19]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)
model.to(device) # GPU에서 실행 되도록 한다.

device = cuda


Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=100, bias=True)
  (2): ReLU()
  (3): Linear(in_features=100, out_features=10, bias=True)
)

In [21]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss() # 다중 분류, 이진분류: nn.BCELoss 또는 nn.BCEWithLogitsLoss
optimizer = optim.Adam(model.parameters())

### 회귀모델 주요 손실 함수
- nn.MSELoss (Mean Squared Error)
- nn.L1Loss (Mean Absolute Error)
- nn.HuberLoss (Hubber Loss) : 오차가 작을 때는 MSE처럼 행동하고, 오차가 클 때는 L1처럼 행동하여 두 함수의 장점을 결합함

In [22]:
for params in model.parameters():
    print(params.shape)

torch.Size([100, 784])
torch.Size([100])
torch.Size([10, 100])
torch.Size([10])


In [23]:
epochs = 5
batches = int(len(train_scaled)/32)
for epoch in range(epochs): # 에포크 반복(학습)
    model.train() # 학습(훈련)
    train_loss = 0 # 손실값 초기화
    for i in range(batches): # 1 epoch 실행
        inputs = train_scaled[i*32:(i+1)*32].to(device) # 배치 입력 데이터 준비
        targets = train_target[i*32:(i+1)*32].to(device) # 배치 데이터에 대한 정답 준비
        optimizer.zero_grad() # 이전에 계산된 기울기값 초기화하는 함수
        outputs = model(inputs) # 모델에 입력 데이터 전달
        loss = criterion(outputs, targets) # 손실값 계산
        loss.backward() # 각 층의 모델 파라메터에 대한 기울기(그레디언트)를 계산
        optimizer.step() # 손실 파라메터 업데이트 
        train_loss += loss.item() # 손실값 기록, item(): 파이토치 텐서로부터 값을 추출

    print(f"에포크:{epoch + 1}, 손실:{train_loss/batches:.4f}")

에포크:1, 손실:0.5428
에포크:2, 손실:0.4004
에포크:3, 손실:0.3594
에포크:4, 손실:0.3320
에포크:5, 손실:0.3119


In [24]:
model.eval() # PyTorch에서 모델을 검증(Validation) 및 추론(Inference) 모드로 전환하는 메서드
with torch.no_grad(): # PyTorch에서 자동 미분(Autograd) 엔진을 비활성화, Gradient)를 계산하지 않음
    val_scaled = val_scaled.to(device) # 검증용 데이터 GPU로 전송
    val_target = val_target.to(device) # 검증용 데이터에 대한 정답 GPU로 전송
    outputs = model(val_scaled)
    # torch.argmax(): 1번째 차원(가로 행 방향)을 기준으로 가장 큰 값을 가진 원소의 인덱스(위치)를 반환
    predicts = torch.argmax(outputs, 1)
    corrects = (predicts == val_target).sum().item() # 일치하는 데이터 카운트

accuracy = corrects / len(val_target) # 정확도 계산
print(f"검증 정확도: {accuracy:.4f}")

검증 정확도: 0.8719


In [25]:
torch.save(model.state_dict(), 'fashion_mnist_best.pt') # 모델 저장

In [26]:
model.load_state_dict(torch.load('fashion_mnist_best.pt', weights_only=True)) # 모델 로드

<All keys matched successfully>

In [27]:
model.eval() # PyTorch에서 모델을 검증(Validation) 및 추론(Inference) 모드로 전환하는 메서드
with torch.no_grad(): # PyTorch에서 자동 미분(Autograd) 엔진을 비활성화, Gradient)를 계산하지 않음
    val_scaled = val_scaled.to(device) # 검증용 데이터 GPU로 전송
    val_target = val_target.to(device) # 검증용 데이터에 대한 정답 GPU로 전송
    outputs = model(val_scaled)
    # torch.argmax(): 1번째 차원(가로 행 방향)을 기준으로 가장 큰 값을 가진 원소의 인덱스(위치)를 반환
    predicts = torch.argmax(outputs, 1)
    corrects = (predicts == val_target).sum().item() # 일치하는 데이터 카운트

accuracy = corrects / len(val_target) # 정확도 계산
print(f"검증 정확도: {accuracy:.4f}")

검증 정확도: 0.8719
